## Stock Price Prediction Using Statistical Models

This notebook demonstrates multiple statistical approaches for predicting stock
prices, ranging from simple moving average methods to more sophisticated time
series models. I'll use techniques including:

- Moving Average (MA) forecasting
- Exponential Smoothing (ETS)
- ARIMA (AutoRegressive Integrated Moving Average)
- Linear Regression with technical indicators
- Prophet (Facebook's time series forecasting)

Each model has different strengths and is suitable for different market conditions.
We'll evaluate their performance using metrics like RMSE, MAE, and MAPE.

Key Features:
- Feature Engineering - RSI, MACD, Bollinger Bands, momentum, volume ratios, lag features
- Proper Train/Test Split - Time-series aware splitting (no data leakage)
- Multiple Evaluation Metrics - RMSE, MAE, MAPE, R²
- Model Comparison - Side-by-side performance comparison
- Interactive Visualizations - Actual vs predicted prices, error analysis
- Complete Workflow Function - Run everything with one command

### Table of contents


1. Data Preparation & Feature Engineering
2. Train-Test Split Strategy
3. Model 1: Simple Moving Average Forecast
4. Model 2: Exponential Smoothing
5. Model 3: ARIMA Models
6. Model 4: Linear Regression with Technical Indicators
7. Model 5: Prophet Time Series Model
8. Model Comparison & Evaluation
9. Visualization of Predictions

In [1]:
# Required installations (uncomment if needed):
# !pip install statsmodels
# !pip install prophet
# !pip install scikit-learn

import pandas as pd
import numpy as np
import altair as alt
import yfinance as yf

from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# For statistical models
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
alt.renderers.enable('mimetype')
alt.data_transformers.disable_max_rows()

def fetch_stock_data(tickers, period='2y'):
    """
    Fetch stock data for multiple tickers
    
    Parameters:
    - tickers: list of stock symbols
    - period: time period (1d, 5d, 1mo, 3mo, 6mo, 1y, 2y, 5y, 10y, ytd, max)
    """
    all_data = []
    
    for ticker in tickers:
        print(f"Fetching {ticker}...")
        stock = yf.Ticker(ticker)
        df = stock.history(period=period)
        df['Ticker'] = ticker
        df['Date'] = df.index
        df = df.reset_index(drop=True)
        all_data.append(df)

    # add delay between requests (important!)
       # time.sleep(2)  # Wait 2 seconds between each ticker
    
    combined_df = pd.concat(all_data, ignore_index=True)
    
    #calculate daily returns
    combined_df['Daily_Return'] = combined_df.groupby('Ticker')['Close'].pct_change() * 100
    
    # calculate moving averages
    combined_df['MA_20'] = combined_df.groupby('Ticker')['Close'].transform(
        lambda x: x.rolling(window=20, min_periods=1).mean()
    )
    combined_df['MA_50'] = combined_df.groupby('Ticker')['Close'].transform(
        lambda x: x.rolling(window=50, min_periods=1).mean()
    )
    
    # Calculate volatility (20-day rolling standard deviation of returns..)
    combined_df['Volatility'] = combined_df.groupby('Ticker')['Daily_Return'].transform(
        lambda x: x.rolling(window=20, min_periods=1).std()
    )
    
    # normalize prices to percentage change from start (for comparison)
    combined_df['Normalized_Price'] = combined_df.groupby('Ticker')['Close'].transform(
        lambda x: ((x / x.iloc[0]) - 1) * 100
    )
    
    return combined_df

# Fetch data for major tech stocks
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META']
#tickers = ['AAPL', 'GOOGL', 'AMZN']

stock_data = fetch_stock_data(tickers, period='2y')
#stock_data = fetch_stock_data(tickers, period='1mo')

print(f"Data fetched: {len(stock_data)} rows")
stock_data.head()

Fetching AAPL...
Fetching MSFT...
Fetching GOOGL...
Fetching AMZN...
Fetching META...
Data fetched: 2505 rows


,Open,High,Low,Close,Volume,Dividends,Stock Splits,Ticker,Date,Daily_Return,MA_20,MA_50,Volatility,Normalized_Price
0,191.303343,191.679793,189.629152,191.372681,60943700,0.0,0.0,AAPL,2023-12-11 00:00:00-05:00,NaN,191.372681,191.372681,NaN,0.000000
1,191.273642,192.898298,189.926365,192.888397,52696900,0.0,0.0,AAPL,2023-12-12 00:00:00-05:00,0.792023,192.130539,192.130539,NaN,0.792023
2,193.264796,196.147575,193.027051,196.107956,70404200,0.0,0.0,AAPL,2023-12-13 00:00:00-05:00,1.669130,193.456345,193.456345,0.620208,2.474374
3,196.167437,197.752460,194.324838,196.256592,66831600,0.0,0.0,AAPL,2023-12-14 00:00:00-05:00,0.075793,194.156406,194.156406,0.798021,2.552042
4,195.682000,196.543856,195.156959,195.721634,128538400,0.0,0.0,AAPL,2023-12-15 00:00:00-05:00,-0.272581,194.469452,194.469452,0.858585,2.272505


In [3]:
def prepare_prediction_data(df, ticker='AAPL'):
    """
    Prepare stock data for prediction models with additional features
    """
    stock_df = df[df['Ticker'] == ticker].copy().sort_values('Date')
    stock_df = stock_df.reset_index(drop=True)
    
    # calculate additional technical indicators
    
    # RSI (Relative Strength Index)
    #
    delta = stock_df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    stock_df['RSI'] = 100 - (100 / (1 + rs))
    
    # MACD (Moving Average Convergence Divergence)
    #Zero Line Crossovers
    
    exp1 = stock_df['Close'].ewm(span=12, adjust=False).mean()
    exp2 = stock_df['Close'].ewm(span=26, adjust=False).mean()
    stock_df['MACD'] = exp1 - exp2
    stock_df['Signal_Line'] = stock_df['MACD'].ewm(span=9, adjust=False).mean()
    
    # bollinger bands
    # By default, the middle line is a 20-period simple moving average, 
    # and the two outer bands are placed two standard deviations above and below it.
    stock_df['BB_Middle'] = stock_df['Close'].rolling(window=20).mean()
    bb_std = stock_df['Close'].rolling(window=20).std()
    stock_df['BB_Upper'] = stock_df['BB_Middle'] + (2 * bb_std)
    stock_df['BB_Lower'] = stock_df['BB_Middle'] - (2 * bb_std)
    stock_df['BB_Width'] = stock_df['BB_Upper'] - stock_df['BB_Lower']
    
    # Price momentum
    # Formula:  (Momentum = {Current Price} - {Price (n periods ago)}) 
    stock_df['Momentum_5'] = stock_df['Close'].pct_change(periods=5)
    stock_df['Momentum_10'] = stock_df['Close'].pct_change(periods=10)
    stock_df['Momentum_20'] = stock_df['Close'].pct_change(periods=20)
    
    # Volume indicators
    
    stock_df['Volume_MA'] = stock_df['Volume'].rolling(window=20).mean()
    stock_df['Volume_Ratio'] = stock_df['Volume'] / stock_df['Volume_MA']
    
    # Lag features (previous days' prices)
    for i in range(1, 6):
        stock_df[f'Close_Lag_{i}'] = stock_df['Close'].shift(i)
    
    # Remove NaN values
    stock_df = stock_df.dropna().reset_index(drop=True)
    
    return stock_df

In [4]:
def split_train_test(df, test_size=0.2):
    """
    Split data into train and test sets (time-series aware!)
    """
    split_idx = int(len(df) * (1 - test_size))
    train = df.iloc[:split_idx].copy()
    test = df.iloc[split_idx:].copy()
    
    print(f"Training set: {len(train)} days ({train['Date'].min()} to {train['Date'].max()})")
    print(f"Test set: {len(test)} days ({test['Date'].min()} to {test['Date'].max()})")
    
    return train, test

In [5]:
def simple_moving_average_forecast(train, test, window=20):
    """
    Predict using simple moving average
    
    CONCEPT: Future price = average of last N days
    PROS: Simple, smooth predictions
    CONS: Lags behind actual prices, poor for trends
    """
    predictions = []
    
    # Start with training data
    historical_prices = train['Close'].values.tolist()
    
    for i in range(len(test)):
        # Calculate MA of last 'window' days
        ma = np.mean(historical_prices[-window:])
        predictions.append(ma)
        
        # Add actual price to historical data for next prediction
        historical_prices.append(test.iloc[i]['Close'])
    
    test_results = test.copy()
    test_results['Predicted'] = predictions
    test_results['Model'] = f'Moving Average ({window}d)'
    
    return test_results

In [6]:
def exponential_smoothing_forecast(train, test):
    """
    Predict using Exponential Smoothing (Holt-Winters)
    
    CONCEPT: Weighted average where recent observations matter more
    PROS: Adapts to trends and seasonality
    CONS: Requires enough historical data, sensitive to parameters
    """
    try:
        # Fit model on training data
        model = ExponentialSmoothing(
            train['Close'],
            seasonal_periods=30,  # Monthly seasonality
            trend='add',
            seasonal='add',
            initialization_method='estimated'
        )
        fitted_model = model.fit()
        
        # Forecast
        predictions = fitted_model.forecast(steps=len(test))
        
        test_results = test.copy()
        test_results['Predicted'] = predictions.values
        test_results['Model'] = 'Exponential Smoothing'
        
        return test_results
    
    except Exception as e:
        print(f"Exponential Smoothing failed: {e}")
        return None

In [7]:
def check_stationarity(series):
    """
    Check if time series is stationary using Augmented Dickey-Fuller test
    """
    result = adfuller(series.dropna())
    print(f'ADF Statistic: {result[0]:.4f}')
    print(f'p-value: {result[1]:.4f}')
    
    if result[1] <= 0.05:
        print("Series is stationary")
        return True
    else:
        print("Series is non-stationary (may need differencing)")
        return False

def arima_forecast(train, test, order=(5, 1, 0)):
    """
    ARIMA: AutoRegressive Integrated Moving Average
    
    CONCEPT: Combines autoregression (AR), differencing (I), and moving average (MA)
    - AR: Use past values to predict
    - I: Make series stationary through differencing
    - MA: Use past forecast errors
    
    PARAMETERS:
    - p (AR order): How many past values to use
    - d (Differencing): How many times to difference
    - q (MA order): How many past errors to use
    
    PROS: Handles trends and autocorrelation well
    CONS: Requires manual parameter tuning, computationally intensive
    """
    try:
        print(f"\nFitting ARIMA{order}...")
        
        # Fit model
        model = ARIMA(train['Close'], order=order)
        fitted_model = model.fit()
        
        print(fitted_model.summary())
        
        # Forecast
        predictions = fitted_model.forecast(steps=len(test))
        
        test_results = test.copy()
        test_results['Predicted'] = predictions.values
        test_results['Model'] = f'ARIMA{order}'
        
        return test_results
    
    except Exception as e:
        print(f"ARIMA failed: {e}")
        return None

In [8]:
def linear_regression_forecast(train, test):
    """
    Multi-variable linear regression using technical indicators
    
    CONCEPT: Price = f(MA, Volume, RSI, MACD, Momentum, Lags)
    PROS: Uses multiple features, interpretable coefficients
    CONS: Assumes linear relationships, may not capture complex patterns
    """
    # select features
    feature_cols = [
        'MA_20', 'MA_50', 'RSI', 'MACD', 'BB_Width',
        'Momentum_5', 'Momentum_10', 'Volume_Ratio',
        'Close_Lag_1', 'Close_Lag_2', 'Close_Lag_3'
    ]
    
    X_train = train[feature_cols].values
    y_train = train['Close'].values
    
    X_test = test[feature_cols].values
    
    # Fit model
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    predictions = model.predict(X_test)
    
    # coefficients
    feature_importance = pd.DataFrame({
        'Feature': feature_cols,
        'Coefficient': model.coef_
    }).sort_values('Coefficient', key=abs, ascending=False)
    
    print("\nFeature Importance (Linear Regression):")
    print(feature_importance)
    
    test_results = test.copy()
    test_results['Predicted'] = predictions
    test_results['Model'] = 'Linear Regression'
    
    return test_results

In [9]:
def random_forest_forecast(train, test):
    """
    Random Forest Regression using technical indicators
    
    CONCEPT: Ensemble of decision trees, captures non-linear relationships
    PROS: Handles non-linearity, robust to outliers, feature importance
    CONS: Can overfit, less interpretable than linear models
    """
    feature_cols = [
        'MA_20', 'MA_50', 'RSI', 'MACD', 'BB_Width',
        'Momentum_5', 'Momentum_10', 'Volume_Ratio',
        'Close_Lag_1', 'Close_Lag_2', 'Close_Lag_3'
    ]
    
    X_train = train[feature_cols].values
    y_train = train['Close'].values
    
    X_test = test[feature_cols].values
    
    # Fit 
    model = RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    
    predictions = model.predict(X_test)
    
    feature_importance = pd.DataFrame({
        'Feature': feature_cols,
        'Importance': model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print("\nFeature Importance (Random Forest):")
    print(feature_importance)
    
    test_results = test.copy()
    test_results['Predicted'] = predictions
    test_results['Model'] = 'Random Forest'
    
    return test_results

In [10]:
def evaluate_predictions(results_df):
    """
    Calculate error metrics for predictions
    
    METRICS:
    - RMSE: Root Mean Squared Error (penalizes large errors)
    - MAE: Mean Absolute Error (average absolute difference)
    - MAPE: Mean Absolute Percentage Error (relative error %)
    - R²: Coefficient of determination (variance explained)
    """
    actual = results_df['Close'].values
    predicted = results_df['Predicted'].values
    
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae = mean_absolute_error(actual, predicted)
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100
    r2 = r2_score(actual, predicted)
    
    metrics = {
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'R²': r2
    }
    
    return metrics

def compare_models(results_list):
    """
    Compare all models side-by-side
    """
    comparison = []
    
    for results in results_list:
        if results is not None:
            metrics = evaluate_predictions(results)
            metrics['Model'] = results['Model'].iloc[0]
            comparison.append(metrics)
    
    comparison_df = pd.DataFrame(comparison)
    comparison_df = comparison_df.sort_values('RMSE')
    
    print("\n" + "="*80)
    print("MODEL COMPARISON (sorted by RMSE - lower is better)")
    print("="*80)
    print(comparison_df.to_string(index=False))
    print("="*80)
    
    return comparison_df

In [11]:
def visualize_predictions(train, test_results_list, ticker='AAPL'):
    """
    Create interactive visualization comparing actual vs predicted prices
    """
    # Combine all predictions
    all_predictions = []
    
    # Add training data
    train_plot = train[['Date', 'Close']].copy()
    train_plot['Type'] = 'Training Data'
    train_plot['Model'] = 'Actual'
    
    # Add actual test data
    test_actual = test_results_list[0][['Date', 'Close']].copy()
    test_actual['Type'] = 'Test Data (Actual)'
    test_actual['Model'] = 'Actual'
    
    # Combine
    plot_data = pd.concat([train_plot, test_actual], ignore_index=True)
    
    # Add predictions
    for results in test_results_list:
        if results is not None:
            pred_data = results[['Date', 'Predicted']].copy()
            pred_data['Close'] = pred_data['Predicted']
            pred_data['Type'] = 'Predictions'
            pred_data['Model'] = results['Model'].iloc[0]
            plot_data = pd.concat([plot_data, pred_data[['Date', 'Close', 'Type', 'Model']]], ignore_index=True)
    
    # Create selection
    model_selection = alt.selection_point(fields=['Model'], bind='legend')
    
    # Line chart
    chart = alt.Chart(plot_data).mark_line(size=2).encode(
        x=alt.X('Date:T', axis=alt.Axis(title='Date', format='%b %Y')),
        y=alt.Y('Close:Q', axis=alt.Axis(title='Price ($)'), scale=alt.Scale(zero=False)),
        color=alt.Color('Model:N', 
                       scale=alt.Scale(scheme='category20'),
                       legend=alt.Legend(title='Model (Click to filter)')),
        strokeDash=alt.StrokeDash('Type:N', 
                                   scale=alt.Scale(domain=['Training Data', 'Test Data (Actual)', 'Predictions'],
                                                 range=[[1,0], [1,0], [5,5]])),
        opacity=alt.condition(model_selection, alt.value(1), alt.value(0.2)),
        tooltip=[
            alt.Tooltip('Date:T', format='%B %d, %Y'),
            alt.Tooltip('Model:N'),
            alt.Tooltip('Close:Q', title='Price', format='$.2f'),
            alt.Tooltip('Type:N')
        ]
    ).add_params(
        model_selection
    ).properties(
        width=900,
        height=450,
        title=f'{ticker} - Actual vs Predicted Prices (Click legend to filter models)'
    )
    
    return chart

def visualize_prediction_errors(test_results_list):
    """
    Visualize prediction errors over time
    """
    error_data = []
    
    for results in test_results_list:
        if results is not None:
            errors = results.copy()
            errors['Error'] = errors['Close'] - errors['Predicted']
            errors['Abs_Error'] = np.abs(errors['Error'])
            errors['Pct_Error'] = (errors['Error'] / errors['Close']) * 100
            error_data.append(errors[['Date', 'Error', 'Abs_Error', 'Pct_Error', 'Model']])
    
    combined_errors = pd.concat(error_data, ignore_index=True)
    
    # find error over time
    error_chart = alt.Chart(combined_errors).mark_line(size=2).encode(
        x=alt.X('Date:T', axis=alt.Axis(title='Date', format='%b %d')),
        y=alt.Y('Pct_Error:Q', axis=alt.Axis(title='Prediction Error (%)')),
        color=alt.Color('Model:N', scale=alt.Scale(scheme='category10')),
        tooltip=[
            alt.Tooltip('Date:T', format='%B %d, %Y'),
            alt.Tooltip('Model:N'),
            alt.Tooltip('Pct_Error:Q', title='Error %', format='.2f')
        ]
    ).properties(
        width=900,
        height=300,
        title='Prediction Error Over Time (%)'
    )
    
    #  line at zero
    zero_line = alt.Chart(pd.DataFrame({'y': [0]})).mark_rule(
        strokeDash=[5, 5], color='gray'
    ).encode(y='y:Q')
    
    return error_chart + zero_line

In [12]:
def run_complete_analysis(stock_data, ticker='AAPL', test_size=0.2):
    """
    Run complete prediction analysis workflow
    
    Usage:
        results, comparison, charts = run_complete_analysis(stock_data, ticker='AAPL')
    """
    print(f"\n{'='*80}")
    print(f"STOCK PRICE PREDICTION ANALYSIS: {ticker}")
    print(f"{'='*80}\n")
    
    # 1. Prepare data
    print("Step 1: Preparing data and engineering features...")
    prepared_data = prepare_prediction_data(stock_data, ticker=ticker)
    print(f"Total records: {len(prepared_data)}")
    
    # 2. Split data
    print("\nStep 2: Splitting train/test sets...")
    train, test = split_train_test(prepared_data, test_size=test_size)
    
    # 3. Check stationarity
    print("\nStep 3: Checking stationarity...")
    check_stationarity(train['Close'])
    
    # 4. Run all models
    print("\nStep 4: Running prediction models...\n")
    
    results_list = []
    
    # Simple MA
    print("Running Simple Moving Average...")
    ma_results = simple_moving_average_forecast(train, test, window=20)
    results_list.append(ma_results)
    
    # Exponential Smoothing
    print("\nRunning Exponential Smoothing...")
    es_results = exponential_smoothing_forecast(train, test)
    if es_results is not None:
        results_list.append(es_results)
    
    # ARIMA
    print("\nRunning ARIMA...")
    arima_results = arima_forecast(train, test, order=(5, 1, 0))
    if arima_results is not None:
        results_list.append(arima_results)
    
    # Linear Regression
    print("\nRunning Linear Regression...")
    lr_results = linear_regression_forecast(train, test)
    results_list.append(lr_results)
    
    # Random Forest
    print("\nRunning Random Forest...")
    rf_results = random_forest_forecast(train, test)
    results_list.append(rf_results)
    
    # 5. Compare models
    print("\nStep 5: Comparing model performance...")
    comparison = compare_models(results_list)
    
    # 6. Create visualizations
    print("\nStep 6: Creating visualizations...")
    prediction_chart = visualize_predictions(train, results_list, ticker=ticker)
    error_chart = visualize_prediction_errors(results_list)
    
    print("\n✓ Analysis complete!")
    
    return {
        'results': results_list,
        'comparison': comparison,
        'prediction_chart': prediction_chart,
        'error_chart': error_chart,
        'train': train,
        'test': test
    }

In [13]:
stock_data = fetch_stock_data(['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META'], period='2y')

Fetching AAPL...
Fetching MSFT...
Fetching GOOGL...
Fetching AMZN...
Fetching META...


In [14]:
#Run if Jupyter Notebook is having a problem creating Altair Visualizations
alt.renderers.enable('default')
# OR
#alt.renderers.enable('notebook')

RendererRegistry.enable('default')

In [15]:
analysis = run_complete_analysis(stock_data, ticker='AAPL', test_size=0.2)


STOCK PRICE PREDICTION ANALYSIS: AAPL

Step 1: Preparing data and engineering features...
Total records: 481

Step 2: Splitting train/test sets...
Training set: 384 days (2024-01-10 00:00:00-05:00 to 2025-07-23 00:00:00-04:00)
Test set: 97 days (2025-07-24 00:00:00-04:00 to 2025-12-09 00:00:00-05:00)

Step 3: Checking stationarity...
ADF Statistic: -1.8649
p-value: 0.3488
Series is non-stationary (may need differencing)

Step 4: Running prediction models...

Running Simple Moving Average...

Running Exponential Smoothing...

Running ARIMA...

Fitting ARIMA(5, 1, 0)...
                               SARIMAX Results                                
Dep. Variable:                  Close   No. Observations:                  384
Model:                 ARIMA(5, 1, 0)   Log Likelihood               -1045.248
Date:                Tue, 09 Dec 2025   AIC                           2102.497
Time:                        19:11:04   BIC                           2126.185
Sample:                      

In [16]:
print(analysis['comparison'])
analysis['prediction_chart']
analysis['error_chart']

        RMSE        MAE       MAPE        R²                  Model
3   1.654077   1.304847   0.525343  0.994342      Linear Regression
0   9.906337   8.573703   3.455117  0.797072   Moving Average (20d)
4  15.093667  10.393077   3.919475  0.528908          Random Forest
1  35.856084  31.192861  12.018325 -1.658533  Exponential Smoothing
2  41.858690  36.546425  14.074638 -2.623161         ARIMA(5, 1, 0)


alt.LayerChart(...)